In [3]:
# @title 🌿 Cloud-Connected Smart Plant Care & Watering System (run this single cell)
"""
Everything in ONE cell:
  Virtual IoT sensors (Python threads) --HTTP/JSON--> Flask REST API (cloud backend)
  -> SQLite (users/devices/sensor_readings/watering_events/alerts)
  -> Automation engine (threshold + tank + cooldown + max duration)
  -> Alerts + offline (heartbeat) detection -> live HTML dashboard (Chart.js)
Re-running the cell safely restarts everything. No hardware, no API keys needed.
"""
import os, sys, re, time, math, random, sqlite3, threading, secrets, logging, subprocess, collections, json
from datetime import datetime, timezone
try:
    import flask, requests
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "flask", "requests"])
import requests
from flask import Flask, request, jsonify, Response, abort
from werkzeug.serving import make_server
from werkzeug.exceptions import HTTPException

# ---------- restart handling (re-run cell = clean restart) ----------
if "_RT" in globals():
    _RT["stop"].set()
    try: _RT["srv"].shutdown()
    except Exception: pass
    time.sleep(1.5)
_RT = {"stop": threading.Event()}

# ---------- configuration (env-overridable; no secrets hardcoded) ----------
PORT        = int(os.environ.get("PLANT_PORT", 8050))
DB_PATH     = "/tmp/smart_plant.db"
TICK        = 2.0      # seconds between simulated sensor readings
OFFLINE_AFTER = 8.0    # seconds without data -> device OFFLINE
COOLDOWN    = 25.0     # seconds between automatic waterings
MAX_PUMP_S  = 14.0     # hard safety limit for automatic watering
MANUAL_S    = 6.0      # manual watering pulse length
TARGET_MARGIN = 15     # stop pump at threshold + 15 %
TANK_MIN    = 10.0     # % below which automatic watering is blocked
TANK_ML     = 2500.0   # tank capacity (ml)
ML_PER_S    = 40.0     # pump flow
TANK_PCT_S  = ML_PER_S / TANK_ML * 100
TEMP_ALERT  = 35.0
API_KEY     = os.environ.get("PLANT_API_KEY") or secrets.token_hex(8)       # device auth
USER_TOKEN  = os.environ.get("PLANT_USER_TOKEN") or secrets.token_urlsafe(12)  # dashboard/user auth
PROFILES    = {"SUCCULENT": 20, "TOMATO": 40, "HERB": 35, "INDOOR": 30}
logging.getLogger("werkzeug").setLevel(logging.ERROR)

LOG = collections.deque(maxlen=60)
def now(): return time.time()
def clamp(v, lo, hi): return max(lo, min(hi, v))
def log(msg, lvl="INFO"): LOG.appendleft({"ts": now(), "lvl": lvl, "msg": msg})

# ---------- cloud database (SQLite) ----------
if os.path.exists(DB_PATH): os.remove(DB_PATH)
_lock = threading.RLock()
db = sqlite3.connect(DB_PATH, check_same_thread=False)
db.row_factory = sqlite3.Row
db.executescript("""
CREATE TABLE users(user_id INTEGER PRIMARY KEY, name TEXT, email TEXT, created_at REAL);
CREATE TABLE devices(device_id TEXT PRIMARY KEY, user_id INTEGER, plant_name TEXT, plant_type TEXT, location TEXT,
  moisture_threshold REAL, auto_water INTEGER DEFAULT 1, tank_level REAL DEFAULT 100, pump_on INTEGER DEFAULT 0,
  pump_started REAL, pump_trigger TEXT, pump_event_id INTEGER, last_seen REAL, last_watered REAL, created_at REAL);
CREATE TABLE sensor_readings(reading_id INTEGER PRIMARY KEY AUTOINCREMENT, device_id TEXT, soil_moisture REAL,
  temperature REAL, humidity REAL, light_level REAL, timestamp REAL, UNIQUE(device_id, timestamp));
CREATE INDEX idx_readings_dev_ts ON sensor_readings(device_id, timestamp DESC);
CREATE TABLE watering_events(event_id INTEGER PRIMARY KEY AUTOINCREMENT, device_id TEXT, trigger_type TEXT,
  moisture_before REAL, moisture_after REAL, duration REAL, timestamp REAL);
CREATE INDEX idx_events_dev_ts ON watering_events(device_id, timestamp DESC);
CREATE TABLE alerts(alert_id INTEGER PRIMARY KEY AUTOINCREMENT, device_id TEXT, alert_type TEXT, level TEXT,
  message TEXT, status TEXT, created_at REAL);
""")
def sel(sql, args=()):
    with _lock: return [dict(r) for r in db.execute(sql, args).fetchall()]
def ex(sql, args=()):
    with _lock:
        cur = db.execute(sql, args); db.commit(); return cur

ex("INSERT INTO users VALUES(1,'Demo Student','student@example.com',?)", (now(),))
def add_device(did, name, ptype, loc):
    ex("INSERT INTO devices(device_id,user_id,plant_name,plant_type,location,moisture_threshold,created_at) VALUES(?,?,?,?,?,?,?)",
       (did, 1, name, ptype, loc, PROFILES[ptype], now()))
def get_dev(did):
    r = sel("SELECT * FROM devices WHERE device_id=?", (did,)); return r[0] if r else None
def latest(did):
    r = sel("SELECT * FROM sensor_readings WHERE device_id=? ORDER BY timestamp DESC LIMIT 1", (did,)); return r[0] if r else None

# ---------- alerts ----------
def raise_alert(did, typ, level, msg):
    if sel("SELECT 1 FROM alerts WHERE device_id=? AND alert_type=? AND status!='RESOLVED'", (did, typ)): return
    ex("INSERT INTO alerts(device_id,alert_type,level,message,status,created_at) VALUES(?,?,?,?,'OPEN',?)", (did, typ, level, msg, now()))
    log(f"[{level}] {msg}", "ERR" if level == "CRITICAL" else "WARN")
def resolve_alert(did, typ):
    if ex("UPDATE alerts SET status='RESOLVED' WHERE device_id=? AND alert_type=? AND status!='RESOLVED'", (did, typ)).rowcount:
        log(f"✔ {typ} resolved for {did}")

# ---------- automation engine ----------
def start_pump(d, trigger, m):
    eid = ex("INSERT INTO watering_events(device_id,trigger_type,moisture_before,duration,timestamp) VALUES(?,?,?,0,?)",
             (d["device_id"], trigger, m, now())).lastrowid
    ex("UPDATE devices SET pump_on=1,pump_started=?,pump_trigger=?,pump_event_id=? WHERE device_id=?", (now(), trigger, eid, d["device_id"]))
    log(f"💧 Pump ON ({trigger}) {d['device_id']} moisture={m:.1f}%")
def stop_pump(d, m, reason):
    dur = round(now() - d["pump_started"], 1)
    tank = max(0.0, d["tank_level"] - dur * TANK_PCT_S)
    ex("UPDATE watering_events SET moisture_after=?,duration=? WHERE event_id=?", (m, dur, d["pump_event_id"]))
    ex("UPDATE devices SET pump_on=0,last_watered=?,tank_level=? WHERE device_id=?", (now(), tank, d["device_id"]))
    log(f"🛑 Pump OFF {d['device_id']} after {dur}s ({reason}) moisture={m:.1f}%")

def process_reading(did, m, t):
    """Decision logic; returns True if the pump should be running."""
    d = get_dev(did); thr = d["moisture_threshold"]; pump = False
    if d["pump_on"]:
        el = now() - d["pump_started"]; manual = d["pump_trigger"] == "manual"
        limit = MANUAL_S if manual else MAX_PUMP_S
        target = 98 if manual else min(100, thr + TARGET_MARGIN)
        reason = None
        if el >= limit: reason = "manual pulse complete" if manual else "max duration safety limit"
        elif m >= target: reason = "target moisture reached"
        elif d["tank_level"] - el * TANK_PCT_S <= 0: reason = "tank empty"
        if reason: stop_pump(d, m, reason)
        else: pump = True
    elif m < thr:
        raise_alert(did, "LOW_MOISTURE", "CRITICAL" if m < thr - 10 else "WARNING",
                    f"{d['plant_name']} moisture dropped below {thr:.0f}% (now {m:.0f}%).")
        cool_ok = not d["last_watered"] or now() - d["last_watered"] >= COOLDOWN
        if d["auto_water"] and d["tank_level"] > TANK_MIN and cool_ok:
            start_pump(d, "auto", m); pump = True
    else:
        resolve_alert(did, "LOW_MOISTURE")
    if t > TEMP_ALERT:
        raise_alert(did, "HIGH_TEMP", "CRITICAL" if t > 38 else "WARNING", f"{d['plant_name']} temperature exceeded {TEMP_ALERT:.0f}°C (now {t:.1f}°C).")
    else: resolve_alert(did, "HIGH_TEMP")
    tank = get_dev(did)["tank_level"]
    if tank <= TANK_MIN: raise_alert(did, "LOW_TANK", "CRITICAL", f"Water tank for {d['plant_name']} is low ({tank:.0f}%). Automatic watering paused.")
    else: resolve_alert(did, "LOW_TANK")
    return pump

def monitor(stop):  # heartbeat / offline detection
    while not stop.is_set():
        try:
            for d in sel("SELECT * FROM devices WHERE last_seen IS NOT NULL"):
                if now() - d["last_seen"] > OFFLINE_AFTER:
                    raise_alert(d["device_id"], "DEVICE_OFFLINE", "CRITICAL", f"No sensor data received from {d['device_id']} during the expected interval.")
                    if d["pump_on"]:
                        l = latest(d["device_id"]); stop_pump(d, l["soil_moisture"] if l else 0, "failsafe: device offline")
        except Exception as e: log(f"monitor error: {e}", "ERR")
        stop.wait(1)

# ---------- virtual IoT sensor ----------
class Sim:
    def __init__(s, did, ptype, m0):
        s.id, s.m, s.temp, s.hum, s.light = did, m0, 24 + random.uniform(-2, 2), 60.0, 0.0
        s.clock = random.uniform(6 * 60, 12 * 60)           # simulated minutes of day
        s.decay = {"SUCCULENT": .55, "TOMATO": 1.0, "HERB": .85, "INDOOR": .7}.get(ptype, .8)
        s.pump = s.offline = False; s.heat = 0
    def step(s):
        s.clock = (s.clock + 15) % 1440; hr = s.clock / 60
        sun = max(0, math.sin(math.pi * (hr - 6) / 12))       # day/night light cycle
        s.light = clamp(sun * 100 + (random.uniform(-3, 3) if sun > 0 else 0), 0, 100)
        target_t = 20 + 12 * sun + (9 if s.heat > 0 else 0)
        s.temp = clamp(s.temp + .25 * (target_t - s.temp) + random.uniform(-.2, .2), 15, 42)
        s.hum = clamp(s.hum + .2 * ((82 - (s.temp - 18) * 1.8) - s.hum) + random.uniform(-.8, .8), 30, 90)
        if s.pump: s.m += random.uniform(4.2, 5.8)            # watering raises moisture
        else: s.m -= s.decay * (0.6 + 0.03 * max(0, s.temp - 15)) * random.uniform(.85, 1.15)  # slow drying
        s.m = clamp(s.m, 0, 100); s.heat = max(0, s.heat - 1)
    def payload(s, ts=None):
        return {"device_id": s.id, "soil_moisture": round(s.m, 1), "temperature": round(s.temp, 1), "humidity": round(s.hum, 1),
                "light_level": round(s.light, 1), "timestamp": ts or datetime.now(timezone.utc).isoformat()}
    def send(s):                                              # REST call with retry + exponential backoff
        body = s.payload()
        for i in range(3):
            try:
                r = requests.post(f"http://127.0.0.1:{PORT}/api/sensors/data", json=body, headers={"X-API-Key": API_KEY}, timeout=3)
                if r.status_code == 200: s.pump = bool(r.json().get("pump")); return True
                if r.status_code < 500: log(f"{s.id}: API rejected reading ({r.status_code})", "WARN"); return False
            except requests.RequestException: pass
            time.sleep(0.4 * 2 ** i)
        log(f"{s.id}: API unreachable after retries", "ERR"); return False

SIMS = {}
def sim_loop(stop):
    while not stop.is_set():
        for s in list(SIMS.values()):
            s.step()
            if not s.offline: s.send()
        stop.wait(TICK)

# ---------- REST API ----------
app = Flask(__name__)
def err(msg, code): return jsonify({"error": msg}), code
def need(did):
    d = get_dev(did)
    if not d: abort(404, "device not found")
    return d
def body(): return request.get_json(silent=True) or {}
def view(d):
    l = latest(d["device_id"]); t = now()
    online = bool(d["last_seen"]) and t - d["last_seen"] <= OFFLINE_AFTER
    st = "OFFLINE" if not online else "WATERING" if d["pump_on"] else "NEEDS WATER" if l and l["soil_moisture"] < d["moisture_threshold"] else "HEALTHY"
    s = SIMS.get(d["device_id"])
    return {**d, "latest": l, "online": online, "status": st, "now": t, "sim_offline": bool(s and s.offline)}

@app.before_request
def guard():
    if request.path.startswith("/api/") and request.path != "/api/sensors/data" and request.headers.get("X-User-Token") != USER_TOKEN:
        return err("unauthorized", 401)
@app.errorhandler(Exception)
def boom(e):
    if isinstance(e, HTTPException): return err(e.description, e.code)
    log(f"server error: {e}", "ERR"); return err("internal server error", 500)

def num(p, k, lo, hi, req=True):
    v = p.get(k)
    if v is None and not req: return None
    if isinstance(v, bool) or not isinstance(v, (int, float)) or math.isnan(v) or not lo <= v <= hi:
        raise ValueError(f"{k} must be a number between {lo} and {hi}")
    return float(v)

@app.post("/api/sensors/data")
def ingest():
    if request.headers.get("X-API-Key") != API_KEY: return err("invalid device API key", 401)
    p = request.get_json(silent=True)
    if not isinstance(p, dict): return err("JSON body required", 400)
    did = p.get("device_id")
    if not isinstance(did, str) or not get_dev(did): return err("unknown device", 404)
    try:
        m = num(p, "soil_moisture", 0, 100); t = num(p, "temperature", -10, 60); h = num(p, "humidity", 0, 100)
        l = num(p, "light_level", 0, 100, req=False)
        ts = datetime.fromisoformat(p["timestamp"]).timestamp() if p.get("timestamp") else now()
    except (ValueError, TypeError, KeyError) as e: return err(f"validation failed: {e}", 400)
    try: ex("INSERT INTO sensor_readings(device_id,soil_moisture,temperature,humidity,light_level,timestamp) VALUES(?,?,?,?,?,?)", (did, m, t, h, l, ts))
    except sqlite3.IntegrityError: return err("duplicate reading", 409)
    ex("UPDATE devices SET last_seen=? WHERE device_id=?", (now(), did))
    resolve_alert(did, "DEVICE_OFFLINE")
    return jsonify({"ok": True, "pump": process_reading(did, m, t)})

@app.get("/api/devices")
def devices(): return jsonify([view(d) for d in sel("SELECT * FROM devices ORDER BY created_at")])
@app.post("/api/devices")
def create_device():
    b = body(); did = str(b.get("device_id", "")).strip(); pt = str(b.get("plant_type", "INDOOR")).upper()
    if not re.fullmatch(r"[A-Za-z0-9_-]{3,24}", did): return err("device_id must be 3-24 chars (letters, digits, - _)", 400)
    if pt not in PROFILES: return err(f"plant_type must be one of {list(PROFILES)}", 400)
    if get_dev(did): return err("device already exists", 409)
    add_device(did, str(b.get("plant_name") or did)[:40], pt, str(b.get("location") or "Lab")[:40])
    SIMS[did] = Sim(did, pt, random.uniform(55, 80)); log(f"➕ Device {did} registered")
    return jsonify(view(get_dev(did))), 201
@app.get("/api/devices/<did>")
def device(did): return jsonify(view(need(did)))
@app.get("/api/devices/<did>/latest")
def dev_latest(did):
    need(did); l = latest(did)
    return jsonify(l) if l else err("no readings yet", 404)
@app.get("/api/devices/<did>/history")
def history(did):
    need(did); n = clamp(request.args.get("limit", 100, type=int), 1, 1000)
    return jsonify(sel("SELECT * FROM (SELECT * FROM sensor_readings WHERE device_id=? ORDER BY timestamp DESC LIMIT ?) ORDER BY timestamp", (did, n)))
@app.put("/api/devices/<did>/threshold")
def set_threshold(did):
    need(did); v = body().get("moisture_threshold")
    if isinstance(v, bool) or not isinstance(v, (int, float)) or not 5 <= v <= 90: return err("moisture_threshold must be a number 5-90", 400)
    ex("UPDATE devices SET moisture_threshold=? WHERE device_id=?", (v, did)); log(f"⚙ {did} threshold -> {v:.0f}%")
    return jsonify({"device_id": did, "moisture_threshold": v})
@app.put("/api/devices/<did>/settings")
def settings(did):
    need(did); b = body()
    if "auto_water" in b: ex("UPDATE devices SET auto_water=? WHERE device_id=?", (int(bool(b["auto_water"])), did)); log(f"⚙ {did} auto-water {'ON' if b['auto_water'] else 'OFF'}")
    if "plant_type" in b:
        pt = str(b["plant_type"]).upper()
        if pt not in PROFILES: return err("unknown plant_type", 400)
        ex("UPDATE devices SET plant_type=?,moisture_threshold=? WHERE device_id=?", (pt, PROFILES[pt], did)); log(f"🌱 {did} profile -> {pt} ({PROFILES[pt]}%)")
    return jsonify(view(get_dev(did)))
@app.post("/api/devices/<did>/water")
def water(did):
    d = need(did); l = latest(did)
    if not d["last_seen"] or now() - d["last_seen"] > OFFLINE_AFTER: return err("device offline - cannot send pump command", 409)
    if d["pump_on"]: return err("pump already running", 409)
    if d["tank_level"] <= 0: return err("water tank empty", 409)
    start_pump(d, "manual", l["soil_moisture"] if l else 0)
    return jsonify({"ok": True, "message": "manual watering started"}), 202
@app.post("/api/devices/<did>/refill")
def refill(did):
    need(did); ex("UPDATE devices SET tank_level=100 WHERE device_id=?", (did,)); log(f"🪣 {did} tank refilled"); return jsonify({"ok": True})
@app.get("/api/devices/<did>/watering-history")
def watering_history(did):
    need(did); return jsonify(sel("SELECT * FROM watering_events WHERE device_id=? ORDER BY timestamp DESC LIMIT 30", (did,)))
@app.get("/api/devices/<did>/analytics")
def analytics(did):
    need(did)
    r = sel("SELECT AVG(soil_moisture) a,MIN(soil_moisture) mn,MAX(soil_moisture) mx,AVG(temperature) t,AVG(humidity) h FROM (SELECT * FROM sensor_readings WHERE device_id=? ORDER BY timestamp DESC LIMIT 500)", (did,))[0]
    e = sel("SELECT COUNT(*) c,COALESCE(SUM(duration),0) s,AVG(moisture_after-moisture_before) g FROM watering_events WHERE device_id=?", (did,))[0]
    recent = sel("SELECT COUNT(*) c FROM sensor_readings WHERE device_id=? AND timestamp>?", (did, now() - 120))[0]["c"]
    f = lambda v: None if v is None else round(v, 1)
    return jsonify({"avg_moisture": f(r["a"]), "min_moisture": f(r["mn"]), "max_moisture": f(r["mx"]), "avg_temp": f(r["t"]), "avg_humidity": f(r["h"]),
                    "watering_events": e["c"], "water_used_l": round(e["s"] * ML_PER_S / 1000, 2), "avg_gain": f(e["g"]),
                    "uptime_pct": round(min(100, recent / (120 / TICK) * 100), 0)})
@app.get("/api/alerts")
def alerts():
    st = request.args.get("status")
    if st: return jsonify(sel("SELECT * FROM alerts WHERE status=? ORDER BY created_at DESC LIMIT 40", (st.upper(),)))
    return jsonify(sel("SELECT * FROM alerts ORDER BY (status='RESOLVED'), created_at DESC LIMIT 40"))
@app.put("/api/alerts/<int:aid>/acknowledge")
def ack(aid):
    if not sel("SELECT 1 FROM alerts WHERE alert_id=?", (aid,)): return err("alert not found", 404)
    ex("UPDATE alerts SET status='ACKNOWLEDGED' WHERE alert_id=? AND status='OPEN'", (aid,)); return jsonify({"ok": True})
@app.get("/api/logs")
def logs(): return jsonify(list(LOG))
@app.post("/api/sim/<did>/<action>")          # demo controls for the virtual device
def sim_action(did, action):
    s = SIMS.get(did) or abort(404, "no simulator for device")
    if action == "offline":
        s.offline = not s.offline; s.pump = False; log(f"📴 {did} simulator {'OFFLINE' if s.offline else 'reconnected'}")
    elif action == "dry": s.m = clamp(s.m - 18, 0, 100); log(f"🏜 {did} soil dried out")
    elif action == "heat": s.heat = 20; log(f"🔥 {did} heat wave")
    else: abort(400, "unknown action")
    return jsonify({"ok": True})

# ---------- dashboard (HTML + Chart.js) ----------
HTML = r"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Smart Plant Care & Watering System</title>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.1/dist/chart.umd.min.js"></script>
<style>
:root{
  --bg:#f5f8fb;--panel:#fff;--line:#dce6ee;--text:#17324d;--mut:#6f8192;
  --green:#13a66a;--green2:#0b8f59;--blue:#2387e8;--cyan:#19a8c9;
  --red:#ef4444;--amber:#f2ad24;--shadow:0 8px 24px rgba(35,61,84,.08);
}
*{box-sizing:border-box}
body{margin:0;background:var(--bg);color:var(--text);font:14px/1.45 Inter,system-ui,-apple-system,Segoe UI,Roboto,sans-serif}
button,input,select{font:inherit}
button{cursor:pointer}
.topbar{
  background:linear-gradient(105deg,#078e5b,#19ae67 55%,#35bc70);color:#fff;
  padding:17px 28px;display:flex;align-items:center;justify-content:space-between;gap:20px
}
.brand{display:flex;align-items:center;gap:13px}.brand-icon{font-size:42px;line-height:1}
.brand h1{margin:0;font-size:21px}.brand p{margin:2px 0 0;opacity:.9;font-size:12px}
.top-right{text-align:right;font-size:12px}.cloud{font-weight:700;margin-bottom:5px}
.app{display:grid;grid-template-columns:190px minmax(0,1fr);min-height:calc(100vh - 83px)}
.sidebar{background:#12283c;color:#dceaf4;padding:18px 12px;display:flex;flex-direction:column}
.nav{display:grid;gap:5px}.nav button{
  border:0;background:transparent;color:#dceaf4;text-align:left;padding:10px 12px;border-radius:8px
}.nav button:hover,.nav button.active{background:#10a86a;color:#fff}
.userbox{margin-top:auto;background:#19384f;border-radius:10px;padding:12px;font-size:12px}
.userbox b{display:block;color:#fff;margin-bottom:2px}.logout{margin-top:10px;width:100%;background:#27485e;border:0;color:#fff;padding:8px;border-radius:7px}
.content{padding:18px 20px 12px;min-width:0}
.section-head{display:flex;justify-content:space-between;align-items:end;margin-bottom:12px}
.section-head h2{margin:0;font-size:21px}.mut{color:var(--mut);font-size:12px}
.status-pill{display:inline-flex;align-items:center;gap:5px;border-radius:999px;padding:4px 9px;font-size:11px;font-weight:700;background:#e9f8f1;color:#078957}
.status-pill.off{background:#fff0f0;color:#e33636}.status-pill.warn{background:#fff7df;color:#b77900}.status-pill.water{background:#eaf5ff;color:#1973c9}
.fleet{display:grid;grid-template-columns:repeat(3,minmax(240px,1fr));gap:14px;margin-bottom:14px}
.card{background:var(--panel);border:1px solid var(--line);border-radius:9px;box-shadow:var(--shadow);padding:14px}
.plant-card{cursor:pointer;transition:.15s}.plant-card:hover{transform:translateY(-2px);box-shadow:0 12px 28px rgba(35,61,84,.12)}
.plant-card.selected{outline:2px solid #73d7aa}
.plant-top{display:flex;gap:10px;align-items:center}.plant-art{width:64px;height:64px;border-radius:8px;background:#f2f7f4;display:grid;place-items:center;font-size:39px}
.plant-name{font-size:16px;font-weight:800}.plant-meta{font-size:11px;color:var(--mut);margin-top:3px}
.metric{margin-top:12px}.metric-row{display:flex;justify-content:space-between;font-size:11px;margin-bottom:5px}.bar{height:7px;background:#e6edf2;border-radius:99px;overflow:hidden}.bar i{display:block;height:100%;border-radius:99px;background:linear-gradient(90deg,#44d17e,#0ca667)}
.mini-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:8px;margin-top:12px}.mini{padding:7px 8px;background:#f7fafc;border-radius:7px}.mini span{display:block;color:var(--mut);font-size:10px}.mini b{font-size:12px}
.card-actions{display:grid;grid-template-columns:1fr 1fr;gap:8px;margin-top:12px}
.btn{border:1px solid #cfdde7;background:#fff;color:#29445b;padding:8px 10px;border-radius:7px}.btn:hover{border-color:#77a9c9}.btn.primary{background:#0eaa67;color:#fff;border-color:#0eaa67;font-weight:700}.btn.blue{background:#eaf5ff;color:#1d72bd;border-color:#b9dcfb}
.main-grid{display:grid;grid-template-columns:minmax(0,2fr) minmax(280px,1fr);gap:14px}
.overview{display:grid;gap:12px}.overview-card h3,.panel h3{margin:0 0 11px;font-size:13px}
.overview-row{display:flex;align-items:center;gap:10px;padding:9px 0;border-bottom:1px solid #edf1f4}.overview-row:last-child{border-bottom:0}
.ov-icon{width:31px;height:31px;border-radius:50%;display:grid;place-items:center;background:#eaf8f2;font-size:16px}
.ov-val{margin-left:auto;font-weight:800}
.quick{display:grid;gap:7px}.quick .btn{text-align:left}
.alert-item{border-left:3px solid var(--blue);padding:8px 9px;background:#f7fafc;border-radius:5px;margin-bottom:7px;font-size:12px}.alert-item.warn{border-color:var(--amber)}.alert-item.critical{border-color:var(--red)}
.analytics{display:grid;grid-template-columns:1fr 1fr;gap:8px}.a-box{background:#f7fafc;padding:9px;border-radius:7px}.a-box span{display:block;color:var(--mut);font-size:10px}.a-box b{font-size:16px}
.charts{display:grid;grid-template-columns:1fr 1fr;gap:14px;margin-top:14px}.chart{height:230px;position:relative}
.bottom{display:grid;grid-template-columns:1.25fr 1fr;gap:14px;margin-top:14px}
table{width:100%;border-collapse:collapse;font-size:11px}th,td{padding:8px 5px;text-align:left;border-bottom:1px solid #edf1f4}th{color:var(--mut);font-weight:600}
.log{max-height:190px;overflow:auto;font:11px ui-monospace,SFMono-Regular,Menlo,monospace}.log div{padding:4px 0;border-bottom:1px dashed #e4ebf0}
.selected-head{display:flex;justify-content:space-between;align-items:center;margin:14px 0 8px}.selected-title{font-size:17px;font-weight:800}
.control-strip{display:flex;gap:8px;flex-wrap:wrap;align-items:center}.control-strip input[type=range]{width:160px;accent-color:var(--green)}
.add-form{display:none;margin-top:12px}.form-grid{display:grid;grid-template-columns:repeat(2,1fr);gap:8px}.form-grid input,.form-grid select{width:100%;padding:8px;border:1px solid var(--line);border-radius:7px}
#toast{position:fixed;right:18px;bottom:18px;background:#17324d;color:#fff;padding:10px 14px;border-radius:8px;opacity:0;transform:translateY(8px);transition:.2s;z-index:99;pointer-events:none}#toast.on{opacity:1;transform:none}
.footer{padding:10px 4px;color:#8092a2;font-size:10px;display:flex;justify-content:space-between}
@media(max-width:1100px){.fleet{grid-template-columns:1fr 1fr}.main-grid{grid-template-columns:1fr}.bottom{grid-template-columns:1fr}}
@media(max-width:760px){.app{grid-template-columns:1fr}.sidebar{display:none}.content{padding:12px}.fleet,.charts{grid-template-columns:1fr}.topbar{padding:14px}.top-right{display:none}}
</style>
</head>
<body>
<header class="topbar">
  <div class="brand">
    <div class="brand-icon">🌿</div>
    <div><h1>Smart Plant Care &amp; Watering System</h1><p>Healthy Plants &nbsp;•&nbsp; Smart Monitoring &nbsp;•&nbsp; Automated Care</p></div>
  </div>
  <div class="top-right"><div class="cloud">☁ Cloud Connected</div><div id="clock">—</div></div>
</header>

<div class="app">
  <aside class="sidebar">
    <nav class="nav">
      <button class="active">⌂ &nbsp; Dashboard</button>
      <button onclick="scrollToFleet()">🌱 &nbsp; My Plants</button>
      <button onclick="showAdd()">＋ &nbsp; Add Plant</button>
      <button onclick="scrollToAlerts()">🔔 &nbsp; Alerts <span id="alertCount" style="float:right"></span></button>
      <button onclick="scrollToHistory()">▣ &nbsp; Watering History</button>
      <button onclick="scrollToAnalytics()">▥ &nbsp; Analytics</button>
      <button onclick="scrollToControls()">⚙ &nbsp; Settings</button>
    </nav>
    <div class="userbox">
      <b>🌿 Demo User</b>
      student@example.com
      <div style="margin-top:8px">API Key: <span id="apiKeyShort">••••••••</span></div>
      <div>Devices: <span id="deviceCount">—</span></div>
      <button class="logout" onclick="toast('Demo session — no logout required')">↪ Logout</button>
    </div>
  </aside>

  <main class="content">
    <div class="section-head" id="fleetAnchor">
      <div><h2>Plant Fleet</h2><div class="mut">Live virtual IoT devices and current plant health</div></div>
      <span id="api" class="status-pill">● Connecting…</span>
    </div>

    <div id="fleet" class="fleet"></div>

    <div class="main-grid">
      <section>
        <div class="selected-head">
          <div><div class="selected-title" id="pname">Select a plant</div><div class="mut" id="pmeta">Choose a plant card above</div></div>
          <span id="pstat" class="status-pill">—</span>
        </div>

        <div class="card" id="controlsAnchor">
          <div class="control-strip">
            <button class="btn primary" id="water">💧 Water Now</button>
            <button class="btn" id="refill">🪣 Refill Tank</button>
            <button class="btn" id="dry">☀ Simulate Dry Soil</button>
            <button class="btn" id="heat">🌡 Simulate Heat Wave</button>
            <button class="btn" id="off">📴 Go Offline</button>
            <label class="mut">Auto <input type="checkbox" id="auto" style="vertical-align:middle"></label>
            <label class="mut">Threshold <b id="thrv">—</b></label>
            <input type="range" id="thr" min="5" max="90" value="35">
          </div>
        </div>

        <div class="charts" id="analyticsAnchor">
          <div class="card"><h3>Soil Moisture Trend</h3><div class="chart"><canvas id="cM"></canvas></div></div>
          <div class="card"><h3>Water Usage</h3><div class="chart"><canvas id="cW"></canvas></div></div>
          <div class="card"><h3>Temperature &amp; Humidity</h3><div class="chart"><canvas id="cT"></canvas></div></div>
          <div class="card"><h3>Light Level</h3><div class="chart"><canvas id="cL"></canvas></div></div>
        </div>

        <div class="bottom" id="historyAnchor">
          <div class="card">
            <h3>Recent Watering Events</h3>
            <table><thead><tr><th>Time</th><th>Trigger</th><th>Moisture</th><th>Duration</th></tr></thead><tbody id="hist"></tbody></table>
          </div>
          <div class="card">
            <h3>Live Event Log</h3>
            <div class="log" id="log"></div>
          </div>
        </div>

        <div class="card add-form" id="addf">
          <h3>Register New Virtual Plant</h3>
          <div class="form-grid">
            <input id="nid" placeholder="Device ID e.g. PLANT-004">
            <input id="nname" placeholder="Plant name e.g. Basil">
            <select id="ntype"></select>
            <input id="nloc" placeholder="Location e.g. Balcony">
          </div>
          <div style="margin-top:9px"><button class="btn primary" id="nsave">Create Device</button> <button class="btn" id="ncancel">Cancel</button></div>
        </div>
      </section>

      <aside class="overview">
        <div class="card">
          <h3>System Overview</h3>
          <div class="overview-row"><div class="ov-icon">🌿</div><span>Total Plants</span><b class="ov-val" id="ovPlants">—</b></div>
          <div class="overview-row"><div class="ov-icon">💧</div><span>Water Tank Level</span><b class="ov-val" id="ovTank">—</b></div>
          <div class="overview-row"><div class="ov-icon">⚙</div><span>Pump Status</span><b class="ov-val" id="ovPump">OFF</b></div>
          <div class="overview-row"><div class="ov-icon">⚠</div><span>Active Alerts</span><b class="ov-val" id="ovAlerts">—</b></div>
        </div>

        <div class="card">
          <h3>Quick Actions</h3>
          <div class="quick">
            <button class="btn blue" onclick="showAdd()">＋ &nbsp; Register New Plant</button>
            <button class="btn blue" onclick="waterNow()">💧 &nbsp; Manual Watering</button>
            <button class="btn blue" onclick="dryNow()">☀ &nbsp; Simulate Dry Soil</button>
            <button class="btn blue" onclick="heatNow()">🌡 &nbsp; Simulate Heat Wave</button>
          </div>
        </div>

        <div class="card" id="alertsAnchor">
          <h3>Recent Alerts <span class="mut" style="float:right">Live</span></h3>
          <div id="alerts"></div>
        </div>

        <div class="card">
          <h3>Plant Analytics</h3>
          <div class="analytics" id="tiles"></div>
        </div>

        <div class="card">
          <h3>Environment</h3>
          <div class="overview-row"><span>🌡 Temperature</span><b class="ov-val" id="envT">—</b></div>
          <div class="overview-row"><span>💧 Humidity</span><b class="ov-val" id="envH">—</b></div>
          <div class="overview-row"><span>☀ Light Level</span><b class="ov-val" id="envL">—</b></div>
        </div>
      </aside>
    </div>

    <div class="footer"><span>🌿 Smart Plant Care &amp; Watering System &nbsp; | &nbsp; Growing a greener tomorrow</span><span>Powered by Flask &nbsp;|&nbsp; SQLite &nbsp;|&nbsp; Chart.js</span></div>
  </main>
</div>
<div id="toast"></div>

<script>
const TOKEN="__TOKEN__",PROF=__PROFILES__;
const $=s=>document.querySelector(s);
const esc=s=>String(s??"").replace(/[&<>"']/g,c=>({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"}[c]));
const fmt=ts=>new Date(ts*1000).toLocaleTimeString([],{hour:"2-digit",minute:"2-digit",hour12:true});
const ago=(n,ts)=>{const s=Math.max(0,Math.round(n-ts));return s<60?s+"s ago":Math.floor(s/60)+"m "+s%60+"s ago"};
async function api(p,o={}){const r=await fetch("/api"+p,{...o,headers:{"Content-Type":"application/json","X-User-Token":TOKEN}});const j=await r.json().catch(()=>({}));if(!r.ok)throw new Error(j.error||("HTTP "+r.status));return j}
const post=(p,b)=>api(p,{method:"POST",body:JSON.stringify(b||{})});
const put=(p,b)=>api(p,{method:"PUT",body:JSON.stringify(b||{})});
let tt,busy=false,drag=false,sel=null;
function toast(m){const t=$("#toast");t.textContent=m;t.classList.add("on");clearTimeout(tt);tt=setTimeout(()=>t.classList.remove("on"),2500)}
async function act(fn,msg){try{await fn();if(msg)toast(msg);tick()}catch(e){toast("⚠ "+e.message)}}
const STC={"HEALTHY":"g","NEEDS WATER":"warn","WATERING":"water","OFFLINE":"off"};
const plantEmoji={TOMATO:"🍅",SUCCULENT:"🌵",HERB:"🌿",INDOOR:"🪴"};

Chart.defaults.color="#6f8192";Chart.defaults.borderColor="#e1e9ef";
const base=()=>({responsive:true,maintainAspectRatio:false,animation:false,interaction:{mode:"index",intersect:false},plugins:{legend:{labels:{boxWidth:9,font:{size:10}}}},scales:{x:{ticks:{maxTicksLimit:7,font:{size:9}}},y:{ticks:{font:{size:9}}}}});
const ds=(label,c,fill)=>({label,data:[],borderColor:c,backgroundColor:c+"28",fill:!!fill,tension:.32,pointRadius:0,borderWidth:2});
const cM=new Chart($("#cM"),{type:"line",data:{labels:[],datasets:[ds("Moisture %","#18a765",1),{...ds("Threshold","#f2ad24"),borderDash:[6,4]}]},options:{...base(),scales:{y:{min:0,max:100}}}});
const cW=new Chart($("#cW"),{type:"bar",data:{labels:[],datasets:[{label:"Before %",data:[],backgroundColor:"#f2ad24"},{label:"After %",data:[],backgroundColor:"#18a765"}]},options:{...base(),scales:{y:{min:0,max:100}}}});
const cT=new Chart($("#cT"),{type:"line",data:{labels:[],datasets:[ds("Temperature °C","#ef4444"),ds("Humidity %","#2387e8")]},options:base()});
const cL=new Chart($("#cL"),{type:"line",data:{labels:[],datasets:[ds("Light %","#f2ad24",1)]},options:{...base(),scales:{y:{min:0,max:100}}}});

$("#ntype").innerHTML=Object.keys(PROF).map(k=>`<option>${k}</option>`).join("");

function renderFleet(devs){
  $("#fleet").innerHTML=devs.map(d=>{
    const m=d.latest?d.latest.soil_moisture:0, st=STC[d.status]||"";
    return `<div class="card plant-card ${d.device_id===sel?"selected":""}" data-id="${esc(d.device_id)}">
      <div class="plant-top">
        <div class="plant-art">${plantEmoji[d.plant_type]||"🪴"}</div>
        <div style="min-width:0;flex:1">
          <div class="plant-name">${esc(d.plant_name)} <span class="status-pill ${st}">● ${esc(d.status)}</span></div>
          <div class="plant-meta">Location: ${esc(d.location)}</div>
          <div class="plant-meta">Type: ${esc(d.plant_type)}</div>
        </div>
      </div>
      <div class="metric"><div class="metric-row"><span>💧 Soil Moisture</span><b>${m.toFixed(0)}%</b></div>
        <div class="bar"><i style="width:${Math.max(0,Math.min(100,m))}%"></i></div>
        <div class="mut" style="text-align:right;margin-top:3px">Threshold: ${d.moisture_threshold.toFixed(0)}%</div>
      </div>
      <div class="mini-grid">
        <div class="mini"><span>🌡 Temperature</span><b>${d.latest?d.latest.temperature.toFixed(1):"—"}°C</b></div>
        <div class="mini"><span>💧 Humidity</span><b>${d.latest?d.latest.humidity.toFixed(0):"—"}%</b></div>
        <div class="mini"><span>☀ Light</span><b>${d.latest?d.latest.light_level.toFixed(0):"—"}%</b></div>
      </div>
      <div class="card-actions"><button class="btn primary" data-water="${esc(d.device_id)}">💧 Water Now</button><button class="btn" data-settings="${esc(d.device_id)}">⚙ Settings</button></div>
    </div>`;
  }).join("") + `<div class="card plant-card" id="addbtn" style="display:grid;place-items:center;min-height:250px;color:var(--mut);font-weight:700">＋ Register New Plant</div>`;
  $("#deviceCount").textContent=devs.length;
  $("#ovPlants").textContent=devs.length;
}

function renderDevice(d,h,ev,an){
  if(!d)return;
  const L=d.latest, st=STC[d.status]||"";
  $("#pname").textContent=d.plant_name;
  $("#pmeta").textContent=`${d.device_id} · ${d.plant_type} · ${d.location} · threshold ${d.moisture_threshold.toFixed(0)}%`;
  $("#pstat").textContent=d.status;$("#pstat").className="status-pill "+st;
  $("#auto").checked=!!d.auto_water;
  if(!drag)$("#thr").value=d.moisture_threshold;
  $("#thrv").textContent=d.moisture_threshold.toFixed(0)+"%";
  $("#off").textContent=d.sim_offline?"📶 Reconnect":"📴 Go Offline";

  $("#ovTank").textContent=d.tank_level.toFixed(0)+"% ("+Math.round(d.tank_level*25)+" ml)";
  $("#ovPump").textContent=d.pump_on?"ON":"OFF";$("#ovPump").style.color=d.pump_on?"#2387e8":"#ef4444";
  $("#envT").textContent=L?L.temperature.toFixed(1)+"°C":"—";
  $("#envH").textContent=L?L.humidity.toFixed(0)+"%":"—";
  $("#envL").textContent=L?L.light_level.toFixed(0)+"%":"—";

  const labels=h.map(r=>fmt(r.timestamp));
  cM.data.labels=labels;cM.data.datasets[0].data=h.map(r=>r.soil_moisture);cM.data.datasets[1].data=h.map(()=>d.moisture_threshold);cM.update("none");
  const e2=[...ev].reverse();
  cW.data.labels=e2.map(e=>fmt(e.timestamp));cW.data.datasets[0].data=e2.map(e=>e.moisture_before);cW.data.datasets[1].data=e2.map(e=>e.moisture_after);cW.update("none");
  cT.data.labels=labels;cT.data.datasets[0].data=h.map(r=>r.temperature);cT.data.datasets[1].data=h.map(r=>r.humidity);cT.update("none");
  cL.data.labels=labels;cL.data.datasets[0].data=h.map(r=>r.light_level);cL.update("none");

  const T=(l,v)=>`<div class="a-box"><span>${l}</span><b>${v??"—"}</b></div>`;
  $("#tiles").innerHTML=T("Avg moisture",an.avg_moisture!=null?an.avg_moisture+"%":null)+
    T("Min / Max",an.min_moisture!=null?an.min_moisture+" / "+an.max_moisture+"%":null)+
    T("Avg temp",an.avg_temp!=null?an.avg_temp+"°C":null)+T("Avg humidity",an.avg_humidity!=null?an.avg_humidity+"%":null)+
    T("Watering events",an.watering_events)+T("Water used",an.water_used_l+" L");

  $("#hist").innerHTML=ev.slice(0,8).map(e=>`<tr><td>${fmt(e.timestamp)}</td><td>${e.trigger_type==="auto"?"🤖 Auto":"🖐 Manual"}</td><td>${e.moisture_before.toFixed(0)}% → ${e.moisture_after!=null?e.moisture_after.toFixed(0)+"%":"…"}</td><td>${e.duration?e.duration+"s":"running"}</td></tr>`).join("")||`<tr><td colspan="4" class="mut">No watering events yet</td></tr>`;
}

function renderAlerts(a,n){
  const open=a.filter(x=>x.status==="OPEN").length;
  $("#alertCount").textContent=open?open:"";$("#ovAlerts").textContent=open;
  $("#alerts").innerHTML=a.slice(0,6).map(x=>`<div class="alert-item ${x.level==="CRITICAL"?"critical":x.level==="WARNING"?"warn":""}">
    <b>${esc(x.message)}</b><div class="mut">${esc(x.device_id)} · ${ago(n,x.created_at)}</div>
    ${x.status==="OPEN"?`<button class="btn" style="margin-top:5px;padding:4px 7px" data-ack="${x.alert_id}">Acknowledge</button>`:""}
  </div>`).join("")||`<div class="mut">No active alerts 🎉</div>`;
}
function renderLog(l){$("#log").innerHTML=l.slice(0,40).map(x=>`<div><span class="mut">${fmt(x.ts)}</span> ${esc(x.msg)}</div>`).join("")}

async function tick(){
  if(busy)return;busy=true;
  try{
    const devs=await api("/devices");
    if(!sel||!devs.find(d=>d.device_id===sel))sel=devs[0]?.device_id||null;
    renderFleet(devs);
    const [al,lg]=await Promise.all([api("/alerts"),api("/logs")]);
    if(sel){
      const [h,ev,an]=await Promise.all([
        api(`/devices/${sel}/history?limit=90`),
        api(`/devices/${sel}/watering-history`),
        api(`/devices/${sel}/analytics`)
      ]);
      renderDevice(devs.find(d=>d.device_id===sel),h,ev,an);
    }
    renderAlerts(al,devs.length?devs[0].now:Date.now()/1000);renderLog(lg);
    $("#api").className="status-pill";$("#api").textContent="● API Online";
    $("#clock").textContent="Updated "+new Date().toLocaleString();
  }catch(e){
    $("#api").className="status-pill off";$("#api").textContent="● "+e.message;
  }finally{busy=false}
}

function showAdd(){$("#addf").style.display="block";$("#addf").scrollIntoView({behavior:"smooth",block:"center"})}
function scrollToFleet(){$("#fleetAnchor").scrollIntoView({behavior:"smooth"})}
function scrollToAlerts(){$("#alertsAnchor").scrollIntoView({behavior:"smooth",block:"center"})}
function scrollToHistory(){$("#historyAnchor").scrollIntoView({behavior:"smooth",block:"center"})}
function scrollToAnalytics(){$("#analyticsAnchor").scrollIntoView({behavior:"smooth",block:"center"})}
function scrollToControls(){$("#controlsAnchor").scrollIntoView({behavior:"smooth",block:"center"})}
function waterNow(){if(sel)act(()=>post(`/devices/${sel}/water`),"💧 Watering started")}
function dryNow(){if(sel)act(()=>post(`/sim/${sel}/dry`),"☀ Soil dried out")}
function heatNow(){if(sel)act(()=>post(`/sim/${sel}/heat`),"🌡 Heat wave started")}

document.addEventListener("click",e=>{
  const t=e.target.closest("[data-id],[data-water],[data-settings],[data-ack],#addbtn");
  if(!t)return;
  if(t.id==="addbtn"){showAdd();return}
  if(t.dataset.id){sel=t.dataset.id;tick();return}
  if(t.dataset.water){sel=t.dataset.water;waterNow();return}
  if(t.dataset.settings){sel=t.dataset.settings;scrollToControls();return}
  if(t.dataset.ack)act(()=>put(`/alerts/${t.dataset.ack}/acknowledge`),"Alert acknowledged");
});
$("#auto").onchange=e=>act(()=>put(`/devices/${sel}/settings`,{auto_water:e.target.checked}),"Auto-watering "+(e.target.checked?"enabled":"disabled"));
$("#thr").oninput=e=>{drag=true;$("#thrv").textContent=e.target.value+"%"};
$("#thr").onchange=e=>{drag=false;act(()=>put(`/devices/${sel}/threshold`,{moisture_threshold:+e.target.value}),"Threshold saved")};
$("#water").onclick=waterNow;
$("#refill").onclick=()=>sel&&act(()=>post(`/devices/${sel}/refill`),"Tank refilled");
$("#off").onclick=()=>sel&&act(()=>post(`/sim/${sel}/offline`),"Device connectivity toggled");
$("#dry").onclick=dryNow;$("#heat").onclick=heatNow;
$("#ncancel").onclick=()=>$("#addf").style.display="none";
$("#nsave").onclick=()=>act(async()=>{
  const id=$("#nid").value.trim();
  await post("/devices",{device_id:id,plant_name:$("#nname").value,plant_type:$("#ntype").value,location:$("#nloc").value});
  sel=id;$("#addf").style.display="none";
},"Device registered");

setInterval(tick,2000);tick();
</script>
</body>
</html>"""

@app.get("/")
def index():
    return Response(HTML.replace("__TOKEN__", USER_TOKEN).replace("__PROFILES__", json.dumps(PROFILES)), mimetype="text/html")

# ---------- boot: seed fleet, pre-fill history, start threads ----------
FLEET = [("PLANT-001", "Tomato Plant", "TOMATO", "Greenhouse A", 72),
         ("PLANT-002", "Aloe Vera", "SUCCULENT", "Living Room", 62),
         ("PLANT-003", "Basil", "HERB", "Kitchen Window", 70)]
for did, name, pt, loc, m0 in FLEET:
    add_device(did, name, pt, loc); s = Sim(did, pt, m0); SIMS[did] = s
    for i in range(30):                                      # back-fill ~1 min of history so charts aren't empty
        s.step(); p = s.payload(); ts = now() - (30 - i) * TICK
        ex("INSERT INTO sensor_readings(device_id,soil_moisture,temperature,humidity,light_level,timestamp) VALUES(?,?,?,?,?,?)",
           (did, p["soil_moisture"], p["temperature"], p["humidity"], p["light_level"], ts))
    ex("UPDATE devices SET last_seen=? WHERE device_id=?", (now(), did))
log("🚀 Cloud backend started; virtual IoT fleet online")

srv = make_server("0.0.0.0", PORT, app, threaded=True); _RT["srv"] = srv
threading.Thread(target=srv.serve_forever, daemon=True).start()
for target in (sim_loop, monitor): threading.Thread(target=target, args=(_RT["stop"],), daemon=True).start()

# ---------- quick self-tests (auth, validation, 404s) ----------
time.sleep(1); base_url = f"http://127.0.0.1:{PORT}"; UH = {"X-User-Token": USER_TOKEN}
ok_body = {"device_id": "PLANT-001", "soil_moisture": 50, "temperature": 25, "humidity": 60}
tests = [("Ingest without device key -> 401", requests.post(base_url + "/api/sensors/data", json=ok_body).status_code == 401),
         ("Invalid moisture (150) -> 400", requests.post(base_url + "/api/sensors/data", json={**ok_body, "soil_moisture": 150}, headers={"X-API-Key": API_KEY}).status_code == 400),
         ("Unknown device -> 404", requests.post(base_url + "/api/sensors/data", json={**ok_body, "device_id": "NOPE"}, headers={"X-API-Key": API_KEY}).status_code == 404),
         ("Dashboard API without user token -> 401", requests.get(base_url + "/api/devices").status_code == 401),
         ("List devices -> 3 devices", len(requests.get(base_url + "/api/devices", headers=UH).json()) == 3),
         ("Bad threshold (999) -> 400", requests.put(base_url + "/api/devices/PLANT-001/threshold", json={"moisture_threshold": 999}, headers=UH).status_code == 400)]
for n, okk in tests: print("✅" if okk else "❌", n)

# ---------- show dashboard ----------
print(f"\n🌿 Dashboard live on port {PORT}  |  device key: {API_KEY}")
print("Demo tip: click 'Dry out soil' or wait ~15 s -> low-moisture alert -> pump ON -> moisture recovers -> event logged.\n")
try:
    from google.colab import output
    print("🌿 Embedded dashboard preview:")
    output.serve_kernel_port_as_iframe(PORT, height=1250)
    print("\n🔗 Separate dashboard window:")
    output.serve_kernel_port_as_window(PORT)
except ImportError:
    print(f"Not in Colab -> open http://localhost:{PORT} in your browser (keep this process running).")
    try:
        while True: time.sleep(3600)
    except KeyboardInterrupt: pass

✅ Ingest without device key -> 401
✅ Invalid moisture (150) -> 400
✅ Unknown device -> 404
✅ Dashboard API without user token -> 401
✅ List devices -> 3 devices
✅ Bad threshold (999) -> 400

🌿 Dashboard live on port 8050  |  device key: 3b7a01ea3fe87d60
Demo tip: click 'Dry out soil' or wait ~15 s -> low-moisture alert -> pump ON -> moisture recovers -> event logged.

🌿 Embedded dashboard preview:


<IPython.core.display.Javascript object>


🔗 Separate dashboard window:
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>